# Joint DP: Cross-Bin Segment-Level Mixed-Precision Quantization

**Goal**: Replace the 2-step approach (per-bin segment DP + outer water-filling) with a SINGLE DP that shares budget across all zeta bins simultaneously.

## Existing 2-Step Approach
1. For each bin j, solve segment DP independently with budget c_j
2. Outer water-filling allocates {c_j} across bins

## New Joint DP Approach
- State: (j, m, c) -- bin index, block position, remaining TOTAL budget
- All bins share the same encoder architecture (M=32 FC blocks)
- Budget sharing happens naturally inside the DP via population weighting
- Memory: O(M x C) per bin (sequential processing), NOT O(K x M x C)

---
### Execution
1. **Cell 1-2**: Environment setup (Colab GPU)
2. **Cell 3**: Run joint DP evaluation (GPU, ~10-15 min)
3. **Cell 4-5**: Comparison plot + summary table

## Cell 1: Drive Mount & Environment Setup

In [ ]:
import os, sys

PROJECT_ROOT = "/content/drive/MyDrive/MambaCompression"
MAMBAIC_ROOT = os.path.join(PROJECT_ROOT, "MambaIC")

if not os.path.isdir(PROJECT_ROOT):
    from google.colab import drive
    drive.mount('/content/drive')

assert os.path.isdir(MAMBAIC_ROOT), f"MambaIC not found: {MAMBAIC_ROOT}"
os.chdir(MAMBAIC_ROOT)
print(f"Working directory: {os.getcwd()}")

# Run setup_colab.py for VMamba CUDA kernel + dependencies
setup_path = os.path.join(PROJECT_ROOT, "setup_colab.py")
if os.path.isfile(setup_path):
    print("Running setup_colab.py ...")
    exec(open(setup_path).read())
else:
    !pip install -q einops scipy tqdm thop fvcore pybind11

!pip install -q seaborn compressai timm pulp 2>/dev/null | tail -1
print("\n=== Environment Ready ===")

: 

## Cell 2: Verify Prerequisites

Check that all cached CSVs from prior pipeline runs exist.

In [ ]:
import torch
import pandas as pd
import numpy as np

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {mem_gb:.1f} GB")

csv_dir = "results/csv"
required_csvs = [
    "segment_dp_omegas.csv",
    "rpmpq_v2_zeta.csv",
    "rpmpq_v2_perfect_rates.csv",
]
kappa_names = ["rpmpq_v2_step1_nmse_kappa.csv", "rpmpq_v2_kappa.csv"]

all_ok = True
for f in required_csvs:
    exists = os.path.exists(f"{csv_dir}/{f}")
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {f}")
    if not exists:
        all_ok = False

kappa_ok = any(os.path.exists(f"{csv_dir}/{k}") for k in kappa_names)
print(f"  [{'OK' if kappa_ok else 'MISSING'}] kappa CSV")
if not kappa_ok:
    all_ok = False

# Optional: existing 2-step outage curves for comparison
outage_csv = f"{csv_dir}/outage_curves_per_bin.csv"
has_2step = os.path.exists(outage_csv)
print(f"  [{'OK' if has_2step else 'OPTIONAL'}] outage_curves_per_bin.csv (for 2-step comparison)")

# Model & data
model_ok = os.path.exists("saved_models/mamba_transnet_L2_dim512_baseline/best.pth")
data_ok = os.path.exists("data/DATA_Htestout.mat")
print(f"  [{'OK' if model_ok else 'MISSING'}] Model checkpoint")
print(f"  [{'OK' if data_ok else 'MISSING'}] Test data")

if all_ok and model_ok and data_ok:
    print("\nAll prerequisites satisfied. Ready to run.")
else:
    print("\nMISSING files. Run rpmpq_v2.py and segment_dp_policy.py first.")

## Cell 3: Run Joint DP Evaluation (GPU)

Compares three methods at each saving level:
1. **Equal**: Single policy from median-bin DP applied to all samples
2. **2-Step**: Per-bin segment DP with equal budget + outer greedy/grid allocation
3. **Joint DP**: Single DP across all bins with shared budget (proposed)

For each (saving, gamma) pair: solve DP, apply per-bin policies via GPU inference, measure outage.

**Estimated time: 10-20 min** depending on GPU.

In [ ]:
import importlib, sys, os

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

# Reload to pick up any code changes
import analysis.joint_dp_outage as jdp
importlib.reload(jdp)
from analysis.joint_dp_outage import run_joint_dp_evaluation

df_results = run_joint_dp_evaluation(
    target_savings_list=None,   # uses CURVE_SAVINGS: 85% to 97% in 0.2% steps
    gammas=[0.99, 0.98, 0.95],
    objective="nmse",
)

print(f"\nResults shape: {df_results.shape}")
print(f"Columns: {list(df_results.columns)}")

  JOINT DP vs 2-STEP COMPARISON
  Test samples: 20000
    Bin 0: 4000 samples (p=0.200)
    Bin 1: 4000 samples (p=0.200)
    Bin 2: 4000 samples (p=0.200)
    Bin 3: 4000 samples (p=0.200)
    Bin 4: 4000 samples (p=0.200)

  Loading cached segment omegas...

  Loading model and data...
  Device: CUDA
[INFO] Building: UE Encoder [mamba-L2] + BS Decoder [transnet-L2]
  Loading existing outage curves from /content/drive/MyDrive/MambaCompression/MambaIC/results/csv/outage_curves_per_bin.csv

  Savings levels: 49
  Gammas: [0.99, 0.98, 0.95]
  DP objective: nmse



Joint DP sweep: 100%|██████████| 49/49 [11:37<00:00, 14.23s/it]


  Saved comparison results -> /content/drive/MyDrive/MambaCompression/MambaIC/results/csv/joint_dp_comparison.csv
  Saved policies -> /content/drive/MyDrive/MambaCompression/MambaIC/results/csv/joint_dp_policies.csv

  SUMMARY

  gamma = 0.99
    Saving |    Equal | 2Step-Eq | 2Step-Alloc | Joint DP | vs Equal | vs 2Step
  ------------------------------------------------------------------------------
      85.0 |   0.4839 |   0.4900 |     0.4900 |   0.4978 |  -0.0139 |  -0.0077
      85.2 |   0.5006 |   0.5068 |     0.4989 |   0.5086 |  -0.0080 |  -0.0019
      85.5 |   0.5184 |   0.5245 |     0.5068 |   0.5193 |  -0.0009 |  +0.0052
      85.8 |   0.5184 |   0.5245 |     0.5246 |   0.5288 |  -0.0103 |  -0.0042
      86.0 |   0.5358 |   0.5403 |     0.5321 |   0.5391 |  -0.0032 |  +0.0013
      86.2 |   0.5526 |   0.5582 |     0.5405 |   0.5480 |  +0.0046 |  +0.0102
      86.5 |   0.5526 |   0.5582 |     0.5582 |   0.5742 |  -0.0216 |  -0.0160
      86.8 |   0.5982 |   0.6036 |     0.6

## Cell 4: Comparison Plot

Three panels (gamma = 0.99, 0.98, 0.95):
- X-axis: Average BOPs Saving (%)
- Y-axis: Population-weighted outage probability
- Lines: Equal (dashed black), 2-Step (blue), Joint DP (red solid)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)

csv_path = "results/csv/joint_dp_comparison.csv"
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows from {csv_path}")

gammas_to_plot = [g for g in [0.99, 0.98, 0.95] if g in df["gamma"].unique()]
n_panels = len(gammas_to_plot)

fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5))
if n_panels == 1:
    axes = [axes]

for ax_idx, gamma in enumerate(gammas_to_plot):
    ax = axes[ax_idx]
    sub = df[df["gamma"] == gamma].sort_values("target_saving")

    # Equal allocation (median-bin DP, single policy for all)
    ax.plot(
        sub["target_saving"], sub["outage_equal"],
        "k--", linewidth=2, label="Equal (single policy)",
    )

    # 2-step with equal budget per bin
    ax.plot(
        sub["target_saving"], sub["outage_2step_equal_budget"],
        color="#1f77b4", linestyle="-.", linewidth=1.5,
        label="2-Step (equal budget/bin)",
    )

    # 2-step with outer allocation (if available)
    has_alloc = sub["outage_2step_alloc"].notna().any()
    if has_alloc:
        sub_alloc = sub[sub["outage_2step_alloc"].notna()]
        ax.plot(
            sub_alloc["target_saving"], sub_alloc["outage_2step_alloc"],
            color="#2ca02c", linestyle="-", linewidth=2,
            label="2-Step + outer alloc",
        )

    # Joint DP (proposed)
    ax.plot(
        sub["target_saving"], sub["outage_joint_dp"],
        color="#d62728", linestyle="-", linewidth=2.5,
        label="Joint DP (proposed)",
    )

    # Shade improvement of Joint DP over Equal
    ax.fill_between(
        sub["target_saving"],
        sub["outage_joint_dp"],
        sub["outage_equal"],
        where=sub["outage_equal"] > sub["outage_joint_dp"],
        alpha=0.12, color="green",
    )

    ax.set_xlabel("Average BOPs Saving (%)", fontsize=11)
    ax.set_ylabel("Population-Weighted Outage", fontsize=11)
    ax.set_title(f"$\\gamma$ = {gamma}", fontsize=13)
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(True, alpha=0.3)
    ax.set_ylim([-0.02, 1.02])

fig.suptitle(
    "Joint DP vs 2-Step Budget Allocation: Outage Comparison",
    fontsize=14, y=1.02,
)
fig.tight_layout()

out_png = "results/plots/joint_dp_comparison.png"
out_pdf = "results/plots/joint_dp_comparison.pdf"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
plt.show()
print(f"Saved: {out_png}")
print(f"Saved: {out_pdf}")

Loaded 147 rows from results/csv/joint_dp_comparison.csv
Saved: results/plots/joint_dp_comparison.png
Saved: results/plots/joint_dp_comparison.pdf


## Cell 5: Summary Table & Per-Bin Budget Breakdown

Key metrics at representative saving levels (87.5%, 90%, 92.5%, 95%).
Also shows how the joint DP distributes budget differently per bin.

In [ ]:
import pandas as pd
import numpy as np
import os

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)

df = pd.read_csv("results/csv/joint_dp_comparison.csv")

# Summary at key saving levels
key_savings = [87.5, 90.0, 92.5, 95.0]
tol = 0.15  # match within +/- 0.15%

print("=" * 90)
print("  SUMMARY: Joint DP vs 2-Step vs Equal Allocation")
print("=" * 90)

for gamma in [0.99, 0.98, 0.95]:
    sub = df[df["gamma"] == gamma]
    print(f"\n  gamma = {gamma}")
    print(f"  {'Saving':>8s} | {'Equal':>8s} | {'2Step-Eq':>8s} | "
          f"{'2Step+Alloc':>10s} | {'Joint DP':>8s} | "
          f"{'vs Equal':>9s} | {'vs 2Step':>9s}")
    print("  " + "-" * 82)

    for s in key_savings:
        rows = sub[(sub["target_saving"] >= s - tol) &
                    (sub["target_saving"] <= s + tol)]
        if len(rows) == 0:
            # Find nearest
            rows = sub.iloc[(sub["target_saving"] - s).abs().argsort()[:1]]
        if len(rows) == 0:
            continue

        row = rows.iloc[0]
        oe = row["outage_equal"]
        o2e = row["outage_2step_equal_budget"]
        o2a = row["outage_2step_alloc"]
        oj = row["outage_joint_dp"]
        dve = oe - oj
        dv2 = o2e - oj

        o2a_str = f"{o2a:.4f}" if not np.isnan(o2a) else "     N/A"
        print(f"  {row['target_saving']:8.1f} | {oe:8.4f} | {o2e:8.4f} | "
              f"{o2a_str:>10s} | {oj:8.4f} | "
              f"{dve:+9.4f} | {dv2:+9.4f}")

# Per-bin budget allocation from Joint DP
print("\n" + "=" * 90)
print("  PER-BIN BUDGET ALLOCATION (Joint DP)")
print("=" * 90)

has_costs = "joint_cost_0" in df.columns
if has_costs:
    for gamma in [0.99]:
        sub = df[df["gamma"] == gamma]
        print(f"\n  gamma = {gamma}")
        print(f"  {'Saving':>8s} | " +
              " | ".join(f"{'Bin ' + str(j):>10s}" for j in range(5)) +
              " | {'Total wt':>10s}")
        print("  " + "-" * 82)

        for s in key_savings:
            rows = sub[(sub["target_saving"] >= s - tol) &
                        (sub["target_saving"] <= s + tol)]
            if len(rows) == 0:
                rows = sub.iloc[(sub["target_saving"] - s).abs().argsort()[:1]]
            if len(rows) == 0:
                continue
            row = rows.iloc[0]
            costs = [row.get(f"joint_cost_{j}", np.nan) for j in range(5)]
            wcosts = [row.get(f"joint_wcost_{j}", np.nan) for j in range(5)]
            total_wc = sum(w for w in wcosts if not np.isnan(w))

            line = f"  {row['target_saving']:8.1f} | "
            line += " | ".join(f"{c:10.6f}" if not np.isnan(c) else "       N/A"
                               for c in costs)
            line += f" | {total_wc:10.6f}"
            print(line)

    # Show which bins get more/less budget
    print("\n  Interpretation:")
    print("  - Higher cost = more bits allocated to that bin (lower saving)")
    print("  - Joint DP can give more budget to 'hard' bins (high zeta)")
    print("    while reducing budget for 'easy' bins (low zeta)")
else:
    print("  No per-bin cost data available.")

# Per-bin policy segmentations (if available)
pol_csv = "results/csv/joint_dp_policies.csv"
if os.path.exists(pol_csv):
    df_pol = pd.read_csv(pol_csv)
    print("\n" + "=" * 90)
    print("  JOINT DP SEGMENTATION POLICIES")
    print("=" * 90)
    for s in [90.0, 92.5]:
        rows = df_pol[(df_pol["target_saving"] >= s - tol) &
                       (df_pol["target_saving"] <= s + tol)]
        if len(rows) > 0:
            print(f"\n  Saving ~{s}%:")
            for _, row in rows.iterrows():
                print(f"    Bin {int(row['bin'])}: {row['segmentation']}  "
                      f"(cost={row['cost']:.6f})")

  SUMMARY: Joint DP vs 2-Step vs Equal Allocation

  gamma = 0.99
    Saving |    Equal | 2Step-Eq | 2Step+Alloc | Joint DP |  vs Equal |  vs 2Step
  ----------------------------------------------------------------------------------
      87.5 |   0.9933 |   0.9934 |     0.6195 |   0.8387 |   +0.1546 |   +0.1547
      90.0 |   1.0000 |   1.0000 |     1.0000 |   0.9841 |   +0.0159 |   +0.0159
      92.5 |   1.0000 |   1.0000 |     1.0000 |   1.0000 |   +0.0000 |   +0.0000
      95.0 |   0.1469 |   1.0000 |     1.0000 |   1.0000 |   -0.8530 |   +0.0000

  gamma = 0.98
    Saving |    Equal | 2Step-Eq | 2Step+Alloc | Joint DP |  vs Equal |  vs 2Step
  ----------------------------------------------------------------------------------
      87.5 |   0.5553 |   0.5567 |     0.2409 |   0.3570 |   +0.1983 |   +0.1997
      90.0 |   1.0000 |   1.0000 |     1.0000 |   0.7800 |   +0.2200 |   +0.2200
      92.5 |   1.0000 |   1.0000 |     1.0000 |   1.0000 |   +0.0000 |   +0.0000
      95.0 |   0.

## Cell 6: Per-Bin Budget Allocation Visualization

Shows how the Joint DP distributes budget differently per bin compared to equal allocation.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)

df = pd.read_csv("results/csv/joint_dp_comparison.csv")

has_costs = "joint_cost_0" in df.columns
if not has_costs:
    print("No per-bin cost data available. Skipping.")
else:
    gammas_to_plot = [g for g in [0.99, 0.98, 0.95] if g in df["gamma"].unique()]
    n_panels = len(gammas_to_plot)

    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5))
    if n_panels == 1:
        axes = [axes]

    colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 5))

    for ax_idx, gamma in enumerate(gammas_to_plot):
        ax = axes[ax_idx]
        sub = df[df["gamma"] == gamma].sort_values("target_saving")

        for j in range(5):
            col = f"joint_cost_{j}"
            if col in sub.columns:
                vals = sub[col].values
                desc = "easy" if j == 0 else ("hard" if j == 4 else "")
                label = f"Bin {j}" + (f" ({desc})" if desc else "")
                ax.plot(
                    sub["target_saving"], vals,
                    "o-", color=colors[j], markersize=2, label=label,
                )

        ax.set_xlabel("Average BOPs Saving (%)", fontsize=11)
        ax.set_ylabel("Per-Bin FC Budget (kappa)", fontsize=11)
        ax.set_title(f"$\\gamma$ = {gamma}", fontsize=13)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    fig.suptitle(
        "Joint DP: Implicit Per-Bin Budget Allocation",
        fontsize=14, y=1.02,
    )
    fig.tight_layout()

    out_path = "results/plots/joint_dp_budget_allocation.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

Saved: results/plots/joint_dp_budget_allocation.png


## Cell 7: Diagnostic — Joint DP Optimality Analysis

Verify that the Joint DP:
1. Respects the total budget constraint (sum_j p_j * cost_j <= fc_budget)
2. Allocates more budget to hard bins (high zeta) and less to easy bins
3. Achieves lower outage than the 2-step approach at the same total budget

In [ ]:
import pandas as pd
import numpy as np
import os

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)

df = pd.read_csv("results/csv/joint_dp_comparison.csv")

# Non-FC cost (load from kappa CSV)
csv_dir = "results/csv"
kappa_csv = f"{csv_dir}/rpmpq_v2_step1_nmse_kappa.csv"
if not os.path.exists(kappa_csv):
    kappa_csv = f"{csv_dir}/rpmpq_v2_kappa.csv"
kdf = pd.read_csv(kappa_csv)
non_fc_blocks = [b for b in kdf["block"].unique() if "fc_part" not in b]
non_fc_cost = sum(
    kdf[(kdf["block"] == bn) & (kdf["bits"] == 16)]["kappa"].values[0]
    for bn in non_fc_blocks
    if len(kdf[(kdf["block"] == bn) & (kdf["bits"] == 16)]) > 0
)
print(f"Non-FC cost: {non_fc_cost:.6f}")

# Check budget constraint
print("\n=== Budget Constraint Verification ===")
print(f"  {'Saving':>8s} | {'FC Budget':>10s} | {'Sum pj*cj':>10s} | {'Slack':>10s} | {'OK?':>4s}")
print("  " + "-" * 52)

has_costs = "joint_cost_0" in df.columns and "joint_wcost_0" in df.columns
if has_costs:
    sub99 = df[df["gamma"] == 0.99].sort_values("target_saving")
    for _, row in sub99.iterrows():
        s = row["target_saving"]
        fc_budget = (1.0 - s / 100.0) - non_fc_cost
        if fc_budget < 0:
            fc_budget = 0.001
        total_wc = sum(
            row.get(f"joint_wcost_{j}", 0) for j in range(5)
            if not np.isnan(row.get(f"joint_wcost_{j}", np.nan))
        )
        slack = fc_budget - total_wc
        ok = "YES" if slack >= -1e-4 else "NO"
        print(f"  {s:8.1f} | {fc_budget:10.6f} | {total_wc:10.6f} | {slack:+10.6f} | {ok:>4s}")
else:
    print("  No per-bin cost data available.")

# Check if hard bins get more budget
print("\n=== Budget Distribution: Easy vs Hard ===")
if has_costs:
    sub99 = df[df["gamma"] == 0.99].sort_values("target_saving")
    key_savings = [87.5, 90.0, 92.5, 95.0]
    for s in key_savings:
        rows = sub99[(sub99["target_saving"] >= s - 0.15) &
                      (sub99["target_saving"] <= s + 0.15)]
        if len(rows) == 0:
            continue
        row = rows.iloc[0]
        costs = [row.get(f"joint_cost_{j}", np.nan) for j in range(5)]
        if all(not np.isnan(c) for c in costs):
            easy_cost = costs[0]
            hard_cost = costs[4]
            ratio = hard_cost / easy_cost if easy_cost > 1e-9 else float("inf")
            print(f"  {s:.1f}%: Bin0(easy)={easy_cost:.6f}  "
                  f"Bin4(hard)={hard_cost:.6f}  ratio={ratio:.2f}x")

# Win/loss count
print("\n=== Joint DP vs Others: Win/Loss Count ===")
for gamma in [0.99, 0.98, 0.95]:
    sub = df[df["gamma"] == gamma]
    n_total = len(sub)
    wins_vs_equal = (sub["outage_joint_dp"] < sub["outage_equal"] - 1e-6).sum()
    ties_vs_equal = ((sub["outage_joint_dp"] - sub["outage_equal"]).abs() <= 1e-6).sum()
    wins_vs_2step = (sub["outage_joint_dp"] < sub["outage_2step_equal_budget"] - 1e-6).sum()
    ties_vs_2step = ((sub["outage_joint_dp"] - sub["outage_2step_equal_budget"]).abs() <= 1e-6).sum()

    print(f"  gamma={gamma}: vs Equal: {wins_vs_equal}W/{ties_vs_equal}T/"
          f"{n_total - wins_vs_equal - ties_vs_equal}L  "
          f"vs 2Step-Eq: {wins_vs_2step}W/{ties_vs_2step}T/"
          f"{n_total - wins_vs_2step - ties_vs_2step}L")

    # Best improvement point
    best_idx = sub["improvement_vs_equal"].idxmax()
    best = sub.loc[best_idx]
    print(f"    Best improvement vs equal at {best['target_saving']:.1f}%: "
          f"{best['improvement_vs_equal']:+.4f} "
          f"({best['outage_equal']:.4f} -> {best['outage_joint_dp']:.4f})")

Non-FC cost: 0.006788

=== Budget Constraint Verification ===
    Saving |  FC Budget |  Sum pj*cj |      Slack |  OK?
  ----------------------------------------------------
      85.0 |   0.143212 |   0.141798 |  +0.001413 |  YES
      85.2 |   0.140712 |   0.139486 |  +0.001225 |  YES
      85.5 |   0.138212 |   0.137175 |  +0.001037 |  YES
      85.8 |   0.135712 |   0.134863 |  +0.000849 |  YES
      86.0 |   0.133212 |   0.132551 |  +0.000661 |  YES
      86.2 |   0.130712 |   0.130239 |  +0.000473 |  YES
      86.5 |   0.128212 |   0.126386 |  +0.001826 |  YES
      86.8 |   0.125712 |   0.124844 |  +0.000868 |  YES
      87.0 |   0.123212 |   0.121762 |  +0.001450 |  YES
      87.2 |   0.120712 |   0.120220 |  +0.000491 |  YES
      87.5 |   0.118212 |   0.116752 |  +0.001459 |  YES
      87.8 |   0.115712 |   0.115211 |  +0.000501 |  YES
      88.0 |   0.113212 |   0.112514 |  +0.000698 |  YES
      88.2 |   0.110712 |   0.109817 |  +0.000895 |  YES
      88.5 |   0.108212 |   

## Cell 8: Cross-Architecture Validation — CLNet & CRNet

RP-MPQ framework가 MT-AE 전용이 아닌 범용 프레임워크임을 보이기 위해,
CLNet과 CRNet에도 outage-based budget allocation을 적용.

**Note**: CLNet/CRNet omega는 global (per-bin 아님). 따라서 DP policy는 bin 무관하게 동일하지만,
outage 평가는 per-bin으로 수행 → budget allocation이 outage를 줄일 수 있음.

**예상 시간: 10-15분** (2 models × saving sweep × inference)

In [5]:
"""
Cross-architecture outage-based budget allocation for CLNet and CRNet.

FIX (2026-03-21): kappa normalization now uses TOTAL encoder FP32 BOPs
(all layers including conv/BN), matching MT-AE's normalization from rpmpq_v2.py.
Previously used FC-only BOPs with wrong act_bits=32, making "90% saving"
mean different things for different models.
"""
import os, sys, re, math, importlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm import tqdm
from collections import namedtuple

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
PROJECT_ROOT = "/content/drive/MyDrive/MambaCompression"
BASELINES_ROOT = os.path.join(PROJECT_ROOT, "baselines")
os.chdir(MAMBAIC_ROOT)
if MAMBAIC_ROOT not in sys.path:
    sys.path.insert(0, MAMBAIC_ROOT)

from train_ae import (
    CsiDataset, quantize_feedback_torch,
    calculate_su_miso_rate_mrt,
)
from analysis.segment_dp_policy import solve_dp
from analysis.budget_allocation_outage import (
    K_BINS, BIT_OPTIONS, ANCHOR_BITS, SNR, AQ_BITS, GAMMAS,
    _load_zeta_and_bins, _load_r_ref,
    optimize_allocation_greedy,
)
from rpmpq_v2 import RESULTS_CSV

def _utils_stub():
    import types
    if 'utils' in sys.modules:
        return
    pkg = types.ModuleType('utils')
    solver = types.ModuleType('utils.solver')
    class _L:
        def info(self, *a, **k): pass
        def debug(self, *a, **k): pass
        def warning(self, *a, **k): pass
    pkg.logger = _L(); pkg.line_seg = '=' * 60
    solver.Result = namedtuple('Result', ('nmse', 'rho', 'epoch'), defaults=(None,)*3)
    for name, mod in [('utils', pkg), ('utils.solver', solver),
                      ('utils.logger', types.ModuleType('utils.logger')),
                      ('utils.init', types.ModuleType('utils.init')),
                      ('utils.statics', types.ModuleType('utils.statics')),
                      ('utils.scheduler', types.ModuleType('utils.scheduler'))]:
        sys.modules[name] = mod
    try:
        torch.serialization.add_safe_globals([solver.Result])
    except AttributeError:
        pass

def _load_module(name, path):
    import importlib.util
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

def load_baseline_model(model_name, device):
    _utils_stub()
    train_set = CsiDataset(os.path.join(MAMBAIC_ROOT, "data", "DATA_Htrainout.mat"), "HT")
    test_set = CsiDataset(os.path.join(MAMBAIC_ROOT, "data", "DATA_Htestout.mat"), "HT",
                           normalization_params=train_set.normalization_params)
    norm_params = train_set.normalization_params
    if model_name.lower() == 'clnet':
        _CLNet = _load_module('clnet_m', os.path.join(BASELINES_ROOT, 'CLNet-master/models/clnet.py')).CLNet
        net = _CLNet(reduction=4).to(device)
        ckpt_path = os.path.join(BASELINES_ROOT, 'CLNet-master/checkpoints/out4.pth')
        state = torch.load(ckpt_path, map_location=device, weights_only=False)
        net.load_state_dict(state.get('state_dict', state), strict=False)
    elif model_name.lower() == 'crnet':
        _CRNet = _load_module('crnet_m', os.path.join(BASELINES_ROOT, 'CRNet-master/models/crnet.py')).CRNet
        net = _CRNet(reduction=4).to(device)
        ckpt_path = os.path.join(BASELINES_ROOT, 'CRNet-master/checkpoints/out_04.pth')
        state = torch.load(ckpt_path, map_location=device, weights_only=False)
        net.load_state_dict(state.get('state_dict', state), strict=False)
    else:
        raise ValueError(f"Unknown: {model_name}")
    net.eval()
    print(f"  Loaded {model_name} from {ckpt_path}")
    return net, test_set, norm_params

def _count_encoder_params(model_name):
    """Count ALL encoder weight parameters (conv + BN + FC) for a baseline model.

    Mirrors rpmpq_v2.get_encoder_layer_params(): iterates named_modules,
    counts .weight.numel() for every module that has a .weight tensor.

    Returns (total_encoder_params, conv_params_only, fc_params).
    """
    if model_name.lower() == 'crnet':
        # CRNet encoder layers (all modules with .weight under encoder path):
        # encoder1.conv3x3_bn: Conv2d(2,2,3)=36, BN(2)=2
        # encoder1.conv1x9_bn: Conv2d(2,2,[1,9])=36, BN(2)=2
        # encoder1.conv9x1_bn: Conv2d(2,2,[9,1])=36, BN(2)=2
        # encoder2: Conv2d(2,2,3)=36, BN(2)=2
        # encoder_conv.conv1x1_bn: Conv2d(4,2,1)=8, BN(2)=2
        # encoder_fc: Linear(2048,512) weight=1048576
        conv_params = (36+2) + (36+2) + (36+2) + (36+2) + (8+2)  # = 162
        fc_params = 2048 * 512  # = 1048576
    elif model_name.lower() == 'clnet':
        # CLNet Encoder layers (all modules with .weight):
        # encoder1.conv3x3_bn: Conv2d(2,2,3)=36, BN(2)=2
        # encoder1.conv1x9_bn: Conv2d(2,2,[1,9])=36, BN(2)=2
        # encoder1.conv9x1_bn: Conv2d(2,2,[9,1])=36, BN(2)=2
        # sa.spatial: Conv2d(2,1,3)=18, BN(1)=1
        # se.fc[0]: Linear(32,2,bias=False)=64
        # se.fc[2]: Linear(2,32,bias=False)=64
        # encoder2: Conv2d(2,32,1)=64, BN(32)=32
        # encoder_conv.conv1x1_bn: Conv2d(34,2,1)=68, BN(2)=2
        # replace_efc: Conv1d(2048,512,1) weight=1048576
        conv_params = ((36+2) + (36+2) + (36+2)    # encoder1
                       + (18+1)                      # SpatialGate
                       + 64 + 64                     # SELayer
                       + (64+32)                     # encoder2
                       + (68+2))                     # encoder_conv
        # = 427
        fc_params = 2048 * 512  # = 1048576
    else:
        raise ValueError(f"Unknown model: {model_name}")
    total = conv_params + fc_params
    return total, conv_params, fc_params

def get_baseline_fc_info(model_name):
    """Build kappa and segment info for a baseline model.

    Uses the SAME normalization as MT-AE (rpmpq_v2.py):
        bops_fp32 = total_encoder_params * 32 * act_bits   (act_bits=16)
        kappa = seg_params * w_bits * act_bits / bops_fp32
    """
    omega_csv = os.path.join(RESULTS_CSV, f"segment_dp_omegas_{model_name.lower()}.csv")
    df_omega = pd.read_csv(omega_csv)
    segments_set = set()
    omega_global = {}
    for _, row in df_omega.iterrows():
        l, r, b = int(row['l']), int(row['r']), int(row['b'])
        segments_set.add((l, r))
        omega_global[(l, r, b)] = row['omega_nmse']
    segments = sorted(segments_set)
    M = max(r for _, r in segments)

    # --- FIX: use total encoder params, matching rpmpq_v2.py ---
    total_enc_params, conv_params, fc_params = _count_encoder_params(model_name)
    act_bits = 16  # matches rpmpq_v2.py
    total_bops_fp32 = total_enc_params * 32 * act_bits

    print(f"  [{model_name}] Encoder params: total={total_enc_params} "
          f"(conv={conv_params}, fc={fc_params})")
    print(f"  [{model_name}] total_bops_fp32 = {total_bops_fp32:,}")

    kappa_seg = {}
    for (l, r) in segments:
        n_chunks = r - l
        seg_params = n_chunks * fc_params / M
        for b in BIT_OPTIONS:
            kappa_seg[(l, r, b)] = seg_params * b * act_bits / total_bops_fp32

    # --- FIX: non_fc_cost from actual conv params at anchor bits ---
    non_fc_cost = conv_params * ANCHOR_BITS * act_bits / total_bops_fp32

    # Sanity check: FC kappa sum at 16-bit should be ~0.5 (like MT-AE)
    fc_kappa_sum_16 = sum(
        kappa_seg.get((i, i+1, 16), 0) for i in range(M)
    )
    print(f"  [{model_name}] FC kappa sum @16-bit: {fc_kappa_sum_16:.6f} "
          f"(expect ~0.5)")
    print(f"  [{model_name}] non_fc_cost @anchor={ANCHOR_BITS}: {non_fc_cost:.6f}")

    return M, segments, kappa_seg, non_fc_cost, omega_global

def run_baseline_outage_sweep(model_name, device='cuda'):
    print(f"\n{'='*70}")
    print(f"  {model_name}: Outage-Based Budget Allocation Sweep")
    print(f"{'='*70}")
    net, test_set, norm_params = load_baseline_model(model_name, device)
    original_state = {k: v.clone().cpu() for k, v in net.state_dict().items()}
    M, segments, kappa_seg, non_fc_cost, omega_global = get_baseline_fc_info(model_name)
    print(f"  M={M}, Omega entries: {len(omega_global)}")
    zeta_vals, k_indices, zeta_edges = _load_zeta_and_bins()
    r_ref = _load_r_ref()
    N = len(test_set)
    bin_counts = [int(np.sum(k_indices == j)) for j in range(K_BINS)]
    p_bins = [bc / N for bc in bin_counts]
    print(f"  Test samples: {N}")
    fc_param_name = None
    for pname, p in net.named_parameters():
        if ('fc' in pname or 'efc' in pname) and 'weight' in pname and p.shape[0] == 512:
            fc_param_name = pname
            break
    print(f"  FC param: {fc_param_name}")
    assert fc_param_name is not None, f"Could not find FC weight for {model_name}"

    savings_list = np.arange(85.0, 97.01, 0.25).tolist()  # 0.25% step
    results = []
    min_val, range_val = norm_params
    pbar = tqdm(savings_list, desc=f"{model_name} sweep")
    for target_saving in pbar:
        fc_budget = max((1.0 - target_saving / 100.0) - non_fc_cost, 0.001)
        dist, seg = solve_dp(M, segments, omega_global, kappa_seg,
                             fc_budget, BIT_OPTIONS, ANCHOR_BITS)
        if dist == float('inf'):
            for gamma in GAMMAS:
                results.append({'model': model_name, 'target_saving': target_saving,
                                'gamma': gamma, 'outage_equal': 1.0,
                                **{f'outage_bin_{j}': 1.0 for j in range(K_BINS)}})
            continue
        bits_per_chunk = [ANCHOR_BITS] * M
        for (l, r, b) in seg:
            for i in range(l, r):
                bits_per_chunk[i] = b
        net.load_state_dict(original_state)
        with torch.no_grad():
            fc_weight = dict(net.named_parameters())[fc_param_name]
            chunk_size = fc_weight.shape[0] // M
            for i in range(M):
                b = bits_per_chunk[i]
                if b < 16:
                    sl = slice(i * chunk_size, (i + 1) * chunk_size)
                    w = fc_weight.data[sl]
                    n_levels = 2 ** b
                    w_min, w_max = w.min(), w.max()
                    scale = (w_max - w_min) / (n_levels - 1)
                    if scale > 1e-8:
                        fc_weight.data[sl] = ((w - w_min) / scale).round() * scale + w_min
        rates_all = np.zeros(N)
        loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=0)
        idx = 0
        with torch.no_grad():
            for batch in loader:
                bs = batch.shape[0]
                d = batch.to(device)
                x_hat = net(d)
                h_true = (d * range_val) + min_val - 0.5
                h_hat = (x_hat * range_val) + min_val - 0.5
                r = calculate_su_miso_rate_mrt(h_true, h_hat, SNR, device)
                rates_all[idx:idx+bs] = r.cpu().numpy()
                idx += bs
        net.load_state_dict(original_state)
        for gamma in GAMMAS:
            r_ref_N = r_ref[:N]
            outage_per_bin = []
            for j in range(K_BINS):
                idx_j = np.where(k_indices == j)[0]
                o_j = float(np.mean(rates_all[idx_j] < gamma * r_ref_N[idx_j])) if len(idx_j) > 0 else 0.0
                outage_per_bin.append(o_j)
            results.append({
                'model': model_name, 'target_saving': target_saving, 'gamma': gamma,
                'outage_equal': float(np.mean(rates_all < gamma * r_ref_N)),
                **{f'outage_bin_{j}': outage_per_bin[j] for j in range(K_BINS)},
            })
    df = pd.DataFrame(results)
    print(f"\n  Running budget allocation for {model_name}...")
    alloc_results = []
    for gamma in GAMMAS:
        sub = df[df['gamma'] == gamma].sort_values('target_saving')
        sorted_savings = sorted(sub['target_saving'].unique())
        outage_lookup = {}
        for j in range(K_BINS):
            saving_to_outage = dict(zip(sub['target_saving'], sub[f'outage_bin_{j}']))
            outage_lookup[j] = [saving_to_outage.get(s, 1.0) for s in sorted_savings]
        for target_saving in sorted_savings:
            alloc_savings, opt_outage, equal_outage = optimize_allocation_greedy(
                outage_lookup, sorted_savings, p_bins, target_saving)
            improvement = equal_outage - opt_outage
            alloc_results.append({
                'model': model_name, 'target_saving': target_saving, 'gamma': gamma,
                'outage_equal': equal_outage, 'outage_alloc': opt_outage,
                'improvement': improvement,
                'improvement_pct': (improvement / equal_outage * 100) if equal_outage > 1e-9 else 0,
                **{f'saving_{j}': alloc_savings[j] for j in range(K_BINS)},
            })
    df_alloc = pd.DataFrame(alloc_results)
    out_csv = os.path.join(RESULTS_CSV, f"outage_alloc_{model_name.lower()}.csv")
    df_alloc.to_csv(out_csv, index=False)
    print(f"  Saved: {out_csv}")
    print(f"\n  === {model_name} Best Improvements ===")
    for gamma in GAMMAS:
        sub = df_alloc[(df_alloc['gamma'] == gamma) & (df_alloc['improvement'] > 0.001)]
        if len(sub) > 0:
            best = sub.loc[sub['improvement_pct'].idxmax()]
            print(f"  gamma={gamma}: {best['target_saving']:.2f}%  "
                  f"equal={best['outage_equal']:.4f} -> alloc={best['outage_alloc']:.4f}  "
                  f"({best['improvement_pct']:.1f}%)")
    return df_alloc

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device.upper()}\n")
all_baseline_results = []
for model_name in ['CLNet', 'CRNet']:
    try:
        df = run_baseline_outage_sweep(model_name, device)
        all_baseline_results.append(df)
    except Exception as e:
        print(f"\n  ERROR running {model_name}: {e}")
        import traceback; traceback.print_exc()
if all_baseline_results:
    df_all = pd.concat(all_baseline_results, ignore_index=True)
    combined_csv = os.path.join(RESULTS_CSV, "outage_alloc_baselines.csv")
    df_all.to_csv(combined_csv, index=False)
    print(f"\nCombined results saved: {combined_csv}")

Device: CUDA


  CLNet: Outage-Based Budget Allocation Sweep
  Loaded CLNet from /content/drive/MyDrive/MambaCompression/baselines/CLNet-master/checkpoints/out4.pth
  [CLNet] Encoder params: total=1049003 (conv=427, fc=1048576)
  [CLNet] total_bops_fp32 = 537,089,536
  [CLNet] FC kappa sum @16-bit: 0.499796 (expect ~0.5)
  [CLNet] non_fc_cost @anchor=16: 0.000204
  M=32, Omega entries: 531
  Test samples: 20000
  FC param: encoder.replace_efc.weight


CLNet sweep: 100%|██████████| 49/49 [00:34<00:00,  1.40it/s]



  Running budget allocation for CLNet...
  Saved: /content/drive/MyDrive/MambaCompression/MambaIC/results/csv/outage_alloc_clnet.csv

  === CLNet Best Improvements ===
  gamma=0.99: 87.50%  equal=0.9996 -> alloc=0.8987  (10.1%)
  gamma=0.98: 87.50%  equal=0.9137 -> alloc=0.4744  (48.1%)
  gamma=0.95: 87.75%  equal=1.0000 -> alloc=0.2064  (79.4%)

  CRNet: Outage-Based Budget Allocation Sweep
  Loaded CRNet from /content/drive/MyDrive/MambaCompression/baselines/CRNet-master/checkpoints/out_04.pth
  [CRNet] Encoder params: total=1048738 (conv=162, fc=1048576)
  [CRNet] total_bops_fp32 = 536,953,856
  [CRNet] FC kappa sum @16-bit: 0.499923 (expect ~0.5)
  [CRNet] non_fc_cost @anchor=16: 0.000077
  M=32, Omega entries: 531
  Test samples: 20000
  FC param: encoder_fc.weight


CRNet sweep: 100%|██████████| 49/49 [00:30<00:00,  1.63it/s]


  Running budget allocation for CRNet...
  Saved: /content/drive/MyDrive/MambaCompression/MambaIC/results/csv/outage_alloc_crnet.csv

  === CRNet Best Improvements ===
  gamma=0.98: 87.25%  equal=1.0000 -> alloc=0.9859  (1.4%)
  gamma=0.95: 87.25%  equal=0.9996 -> alloc=0.5260  (47.4%)

Combined results saved: /content/drive/MyDrive/MambaCompression/MambaIC/results/csv/outage_alloc_baselines.csv


## Cell 9: Cross-Architecture Comparison Plot

MT-AE (Joint DP) vs CLNet vs CRNet: Equal allocation vs Outage-optimized allocation.
All models on the same figure → **framework generality** 입증.

In [6]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

MAMBAIC_ROOT = "/content/drive/MyDrive/MambaCompression/MambaIC"
os.chdir(MAMBAIC_ROOT)
csv_dir = "results/csv"

# Load all results
df_mtae = pd.read_csv(f"{csv_dir}/joint_dp_comparison.csv") if os.path.exists(f"{csv_dir}/joint_dp_comparison.csv") else None
df_baselines = pd.read_csv(f"{csv_dir}/outage_alloc_baselines.csv") if os.path.exists(f"{csv_dir}/outage_alloc_baselines.csv") else None

if df_mtae is None and df_baselines is None:
    print("No results found. Run Cell 3 and Cell 8 first.")
else:
    model_styles = {
        'MT-AE':  {'color': '#d62728', 'marker': 'o'},
        'CLNet':  {'color': '#2ca02c', 'marker': 's'},
        'CRNet':  {'color': '#1f77b4', 'marker': '^'},
    }

    for gamma in [0.99, 0.98, 0.95]:
        fig, ax = plt.subplots(1, 1, figsize=(8, 5))

        # MT-AE: Joint DP
        if df_mtae is not None and gamma in df_mtae['gamma'].values:
            sub = df_mtae[df_mtae['gamma'] == gamma].sort_values('target_saving')
            st = model_styles['MT-AE']
            ax.plot(sub['target_saving'], sub['outage_equal'],
                    '--', color=st['color'], alpha=0.5, linewidth=1, label='MT-AE equal')
            ax.plot(sub['target_saving'], sub['outage_joint_dp'],
                    '-', color=st['color'], marker=st['marker'], markersize=3,
                    linewidth=2, label='MT-AE Joint DP')

        # CLNet, CRNet
        if df_baselines is not None:
            for model_name in ['CLNet', 'CRNet']:
                sub = df_baselines[(df_baselines['model'] == model_name) &
                                   (df_baselines['gamma'] == gamma)].sort_values('target_saving')
                if len(sub) == 0:
                    continue
                st = model_styles[model_name]
                ax.plot(sub['target_saving'], sub['outage_equal'],
                        '--', color=st['color'], alpha=0.5, linewidth=1,
                        label=f'{model_name} equal')
                ax.plot(sub['target_saving'], sub['outage_alloc'],
                        '-', color=st['color'], marker=st['marker'], markersize=3,
                        linewidth=2, label=f'{model_name} alloc')

        ax.set_xlabel('Average BOPs Saving (%)', fontsize=12)
        ax.set_ylabel('Population-Weighted Outage', fontsize=12)
        ax.set_title(f'Cross-Architecture Budget Allocation ($\\gamma$ = {gamma})', fontsize=13)
        ax.legend(fontsize=8, ncol=2)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([-0.02, 1.02])

        out_path = f"results/plots/cross_arch_outage_g{str(gamma).replace('.','')}.png"
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved: {out_path}")

    # Summary table
    print("\n" + "=" * 80)
    print("  CROSS-ARCHITECTURE SUMMARY: Best Outage Improvement per Model")
    print("=" * 80)
    print(f"  {'Model':>8s} | {'gamma':>5s} | {'Saving':>7s} | {'Equal':>8s} | {'Alloc':>8s} | {'Improve':>8s}")
    print("  " + "-" * 60)

    for model_name in ['MT-AE', 'CLNet', 'CRNet']:
        for gamma in [0.99, 0.98, 0.95]:
            if model_name == 'MT-AE' and df_mtae is not None:
                sub = df_mtae[df_mtae['gamma'] == gamma]
                if len(sub) == 0: continue
                sub = sub.copy()
                sub['improvement'] = sub['outage_equal'] - sub['outage_joint_dp']
                best = sub.loc[sub['improvement'].idxmax()]
                print(f"  {'MT-AE':>8s} | {gamma:5.2f} | {best['target_saving']:6.1f}% | "
                      f"{best['outage_equal']:8.4f} | {best['outage_joint_dp']:8.4f} | "
                      f"{best['improvement']:+8.4f}")
            elif df_baselines is not None:
                sub = df_baselines[(df_baselines['model'] == model_name) &
                                   (df_baselines['gamma'] == gamma) &
                                   (df_baselines['improvement'] > 0.001)]
                if len(sub) == 0: continue
                best = sub.loc[sub['improvement_pct'].idxmax()]
                print(f"  {model_name:>8s} | {gamma:5.2f} | {best['target_saving']:6.1f}% | "
                      f"{best['outage_equal']:8.4f} | {best['outage_alloc']:8.4f} | "
                      f"{best['improvement']:+8.4f}")

Saved: results/plots/cross_arch_outage_g099.png
Saved: results/plots/cross_arch_outage_g098.png
Saved: results/plots/cross_arch_outage_g095.png

  CROSS-ARCHITECTURE SUMMARY: Best Outage Improvement per Model
     Model | gamma |  Saving |    Equal |    Alloc |  Improve
  ------------------------------------------------------------
     MT-AE |  0.99 |   87.2% |   0.9461 |   0.7312 |  +0.2149
     MT-AE |  0.98 |   88.0% |   0.9364 |   0.4396 |  +0.4968
     MT-AE |  0.95 |   90.5% |   0.9836 |   0.6475 |  +0.3361
     CLNet |  0.99 |   87.5% |   0.9996 |   0.8987 |  +0.1009
     CLNet |  0.98 |   87.5% |   0.9137 |   0.4744 |  +0.4393
     CLNet |  0.95 |   87.8% |   1.0000 |   0.2064 |  +0.7936
     CRNet |  0.98 |   87.2% |   1.0000 |   0.9859 |  +0.0141
     CRNet |  0.95 |   87.2% |   0.9996 |   0.5260 |  +0.4736
